# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 55 |
| **Integrantes** | Ronal Mosquera · Federico López Torres · * |
| **Caso de estudio** | Wanderbricks (marketplace de alquiler vacacional) |
| **Fecha de entrega** | domingo 30 de agosto |
| **🎥 Enlace al video** | *(https://docs.google.com/document/d/1kgGL4bqGvjXsOAe8C2TJofR4r_Ha8yaJ/edit?usp=drive_link&ouid=106910370328930537306&rtpof=true&sd=true
pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* 

Enlace Repositorio git: https://github.com/RONAL-MOSQUEREA/bigdata-2026-grupo56


> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.


Wanderbricks es una plataforma de reservas de alojamientos vacacionales (similar a un
marketplace tipo Airbnb) que conecta viajeros (`users`) con anfitriones (`hosts`) que
publican propiedades (`properties`) en distintos destinos (`destinations`). Cada estadía
genera una reserva (`bookings`), uno o más pagos (`payments`) y, con frecuencia, una reseña
posterior (`reviews`). Además, la plataforma registra el comportamiento de navegación de
los usuarios (`clickstream`, `page_views`) y las interacciones con soporte al cliente
(`customer_support_logs`).

El problema de negocio es que esta información nace repartida en fuentes de distinta
naturaleza —tablas transaccionales bien estructuradas, eventos de clickstream con
estructuras anidadas (dispositivo, referrer) y registros de CDC de cambios de estado de
reserva (`booking_updates`)— y no existe hoy una base de datos analítica única que permita
combinarlas de forma confiable para responder preguntas de negocio.

Concretamente, esta base de datos debe responder preguntas como: ¿qué destinos y tipos de
propiedad generan más ingresos y mejor calificación?, ¿cuál es la tasa de cancelación de
reservas y en qué canal o dispositivo se originan más consultas que no se convierten en
reserva?, y ¿cómo evoluciona el ingreso promedio por reserva a lo largo del tiempo? Responder
esto requiere una plataforma que soporte tanto los datos tabulares (reservas, pagos) como los
semiestructurados (clickstream, soporte) bajo un mismo modelo de gobierno.

---
## 2. Descripción de los datos


El catálogo `samples.wanderbricks` contiene siete tablas: `users`, `hosts`, `properties`, `bookings`, `payments`, `reviews` y `clickstream`. A continuación se documenta volumen, tipos, calidad y relaciones.

In [0]:
## 2. Descripción de los datos

#*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.#
#Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,#
#cualquier justificación queda en el aire.*#

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:



# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# Esquema de las tablas que vamos a usar en este trabajo
tablas_clave = [
    "users", "hosts", "properties", "destinations",
    "bookings", "payments", "booking_updates",
    "reviews", "clickstream",
]

for t in tablas_clave:
    print(f"\n=== samples.wanderbricks.{t} ===")
    spark.table(f"samples.wanderbricks.{t}").printSchema()

In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)


#**Descripción de las tablas seleccionadas y sus relaciones**

#- `users`: perfiles de viajeros (nombre, correo, país, tipo de usuario). Clave `user_id`.*
#- `hosts`: perfiles de anfitriones, vinculados a las propiedades que publican.
#- `properties`: listados con título, tipo de propiedad, precio y `destination_id`, que
#  referencia a `destinations`.
#- `bookings`: registro central de reservas — conecta `user_id` con `property_id`, con fechas
#  de check-in/check-out, número de huéspedes, monto total y estado (`status`).
# `payments`: transacciones asociadas a una reserva (`booking_id`), con método y estado de pago.
#- `booking_updates`: **tabla de captura de cambios (CDC)** — cada fila es un cambio de estado##
 # de una reserva a lo largo del tiempo, útil para reconstruir el historial de una reserva.
#- `reviews`: reseñas de huéspedes sobre propiedades, con calificación (`rating`), comentario y
 # una bandera `is_deleted` para borrados lógicos (soft delete) — es necesario filtrarla en
 # cualquier análisis de calificaciones.
#`clickstream`: eventos de navegación (vistas, clics, búsquedas, filtros) con un campo
#  `metadata` **anidado (struct)** que incluye dispositivo y referrer — es la tabla candidata
#  para la sección 4.4 de datos semiestructurados.

#En cuanto a calidad: `reviews` requiere filtrar `is_deleted = false` para no contar reseñas
#eliminadas; `bookings` y `payments` deben tipar explícitamente fechas y montos, que llegan
#en formatos que conviene normalizar en la capa plata. La relación entre tablas sigue un
#Modelo típico de "hechos y dimensiones": `bookings` es la tabla de hechos central, y
#`users`, `properties`, `destinations` y `hosts` actúan como dimensiones.

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| Datos tabulares y transaccionales (`bookings`, `payments`, `users`) con relaciones claras por llave foránea | Fuerte: integridad referencial y joins nativos | Débil: los joins entre colecciones son costosos o inexistentes | Fuerte: soporta SQL y joins sobre archivos Delta | Lakehouse |
| Datos semiestructurados y anidados (`clickstream.metadata`, `customer_support_logs`) | Débil: requiere tablas adicionales para normalizar el anidamiento | Fuerte: pensado para documentos anidados | Fuerte: tipos `struct`/`array` nativos en Spark SQL | Lakehouse |
| Volumen y crecimiento (eventos de clickstream crecen mucho más rápido que las reservas) | Limitado: escalar verticalmente es costoso | Fuerte: escalamiento horizontal nativo | Fuerte: almacenamiento en object storage desacoplado del cómputo | Lakehouse |
| Necesidad de auditoría, control de versiones y reproducibilidad (`booking_updates` como CDC) | Moderado: requiere triggers o tablas de auditoría manuales | Débil: pocas garantías ACID entre documentos | Fuerte: Delta Lake ofrece ACID y *time travel* nativos | Lakehouse |
| Analítica y BI sobre todo el conjunto (ingresos por destino, tasa de cancelación) | Fuerte para SQL, pero no maneja bien lo semiestructurado | Débil para agregaciones SQL complejas | Fuerte: motor Spark SQL sobre los mismos archivos que sirven para BI y ML | Lakehouse |

**Decisión final:** un **lakehouse sobre Delta Lake** (Unity Catalog en Databricks). El caso
Wanderbricks combina datos altamente estructurados (`bookings`, `payments`) con datos
semiestructurados y de alto volumen (`clickstream`, `customer_support_logs`), y necesita
trazabilidad de cambios (`booking_updates`) y consultas analíticas tipo BI sobre todo el
conjunto. Ni un modelo relacional puro (débil frente a lo anidado) ni un NoSQL puro (débil
frente a joins y SQL analítico) cubren ambos frentes; el lakehouse permite un único motor
de consulta, gobierno centralizado con Unity Catalog y garantías ACID + time travel + esquema
evolutivo sobre archivos Parquet/Delta, sin duplicar la infraestructura.

**Referencias (APA 7):**

Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). *Lakehouse: A new generation of open platforms that unify data warehousing and advanced analytics*. Proceedings of CIDR.

Databricks. (2026). *Wanderbricks dataset*. Databricks Documentation. https://docs.databricks.com/aws/en/discover/wanderbricks-dataset

Kleppmann, M. (2017). *Designing data-intensive applications*. O'Reilly Media.

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo56"
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# Ingesta de las tablas del caso a la capa bronce, preservando el esquema de origen
# de forma explícita (una tabla Delta administrada por cada tabla fuente).

tablas_bronce = [
    "users", "hosts", "properties", "destinations",
    "bookings", "payments", "booking_updates",
    "reviews", "clickstream",
]

for t in tablas_bronce:
    df = spark.table(f"samples.wanderbricks.{t}")
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bronze_{t}"))
    print(f"bronze_{t:20s} -> {df.count():>10,} filas | columnas: {len(df.columns)}")

In [0]:
# Verificación del esquema explícito persistido en la capa bronce
spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings").printSchema()

### 4.3 Capa plata — datos limpios y tipados

In [0]:
# TODO: limpieza, tipado y reglas de negocio

from pyspark.sql.functions import col, to_date

# Reservas: tipado explícito de fechas y montos, deduplicación y filtro de nulos críticos
bookings_silver = (
    spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
    .withColumn("check_in", to_date(col("check_in")))
    .withColumn("check_out", to_date(col("check_out")))
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))
    .dropDuplicates(["booking_id"])
    .filter(col("user_id").isNotNull() & col("property_id").isNotNull())
)

(bookings_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings"))

# Reseñas: se excluyen los borrados lógicos (is_deleted = true) y se tipa el rating
reviews_silver = (
    spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_reviews")
    .filter(col("is_deleted") == False)
    .withColumn("rating", col("rating").cast("int"))
)

(reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_reviews"))

print("bookings_silver:", bookings_silver.count(), "filas")
print("reviews_silver:", reviews_silver.count(), "filas")
display(bookings_silver.limit(5))

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas

# Estructura anidada de origen
spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_clickstream").printSchema()

In [0]:
from pyspark.sql.functions import col

clickstream_bronze = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_clickstream")

# Aplanado del struct "metadata" -> columnas planas "device" y "referrer"
clickstream_silver = clickstream_bronze.select(
    "user_id", "event", "property_id", "timestamp",
    col("metadata.device").alias("device"),
    col("metadata.referrer").alias("referrer"),
)

(clickstream_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_clickstream"))

display(clickstream_silver.limit(10))

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
# ACID — una operación que modifique datos
# TODO

# ACID — una operación que modifique datos de forma atómica sobre Delta Lake
TABLA_BOOKINGS = f"{CATALOGO}.{ESQUEMA}.silver_bookings"

id_ejemplo = spark.table(TABLA_BOOKINGS).select("booking_id").limit(1).collect()[0]["booking_id"]

spark.sql(f"""
UPDATE {TABLA_BOOKINGS}
SET status = 'cancelled'
WHERE booking_id = '{id_ejemplo}'
""")

display(spark.table(TABLA_BOOKINGS).filter(f"booking_id = '{id_ejemplo}'"))

In [0]:
# Time travel
# Time travel
display(spark.sql(f"DESCRIBE HISTORY {TABLA_BOOKINGS}"))



In [0]:
# Consulta de una versión anterior (antes del UPDATE) y comparación con la versión actual
version_anterior = (
    spark.sql(f"DESCRIBE HISTORY {TABLA_BOOKINGS}")
    .orderBy("version")
    .limit(1)
    .collect()[0]["version"]
)

df_version_anterior = (
    spark.read.format("delta")
    .option("versionAsOf", version_anterior)
    .table(TABLA_BOOKINGS)
    .filter(f"booking_id = '{id_ejemplo}'")
)

print("Estado en la versión", version_anterior, "(antes del cambio):")
display(df_version_anterior)

print("Estado en la versión actual:")
display(spark.table(TABLA_BOOKINGS).filter(f"booking_id = '{id_ejemplo}'"))

In [0]:
# Evolución de esquema con mergeSchema
# TODO

from pyspark.sql.functions import lit

bookings_con_canal = (
    spark.table(TABLA_BOOKINGS)
    .withColumn("canal_reserva", lit("web"))
)

(bookings_con_canal.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TABLA_BOOKINGS))

spark.table(TABLA_BOOKINGS).printSchema()

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

In [0]:
# Consulta 1 (SQL) — ¿qué destinos generan más ingresos por reservas confirmadas?
display(spark.sql(f"""
SELECT
  d.destination        AS destino,
  COUNT(b.booking_id)  AS reservas,
  ROUND(SUM(b.total_amount), 2) AS ingresos_totales
FROM {CATALOGO}.{ESQUEMA}.silver_bookings AS b
JOIN samples.wanderbricks.properties      AS p ON b.property_id = p.property_id
JOIN samples.wanderbricks.destinations    AS d ON p.destination_id = d.destination_id
WHERE b.status <> 'cancelled'
GROUP BY d.destination
ORDER BY ingresos_totales DESC
LIMIT 10
"""))

In [0]:
# Consulta 2 (PySpark) — propiedades mejor calificadas con al menos 5 reseñas válidas
from pyspark.sql.functions import avg, count, round as pyspark_round, col

properties_df = spark.table("samples.wanderbricks.properties")
reviews_df    = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_reviews")

top_propiedades = (
    properties_df
    .join(reviews_df, properties_df.property_id == reviews_df.property_id)
    .groupBy(properties_df.title.alias("propiedad"), properties_df.property_type)
    .agg(
        pyspark_round(avg(reviews_df.rating), 2).alias("calificacion_promedio"),
        count(reviews_df.rating).alias("numero_resenas"),
    )
    .filter(col("numero_resenas") >= 5)
    .orderBy(col("calificacion_promedio").desc())
)

display(top_propiedades.limit(10))

In [0]:
# Consulta 3 (SQL) — tasa de cancelación de reservas por estado
display(spark.sql(f"""
SELECT
  status,
  COUNT(*) AS total_reservas,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM {CATALOGO}.{ESQUEMA}.silver_bookings
GROUP BY status
ORDER BY total_reservas DESC
"""))

In [0]:
# Consulta 4 (PySpark) — eventos de navegación por tipo de dispositivo (¿dónde conviene invertir en UX?)
from pyspark.sql.functions import count as spark_count

clickstream_silver = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_clickstream")

eventos_por_dispositivo = (
    clickstream_silver
    .groupBy("device", "event")
    .agg(spark_count("*").alias("numero_eventos"))
    .orderBy(col("numero_eventos").desc())
)

display(eventos_por_dispositivo.limit(15))

In [0]:
# Consulta 5 (SQL) — evolución del valor promedio de reserva por mes de check-in
display(spark.sql(f"""
SELECT
  date_format(check_in, 'yyyy-MM')     AS mes,
  COUNT(*)                             AS reservas,
  ROUND(AVG(total_amount), 2)          AS valor_promedio_reserva
FROM {CATALOGO}.{ESQUEMA}.silver_bookings
WHERE check_in IS NOT NULL
GROUP BY date_format(check_in, 'yyyy-MM')
ORDER BY mes
"""))

In [0]:
spark.table("samples.wanderbricks.properties").printSchema()
spark.table("samples.wanderbricks.destinations").printSchema()

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

##  Conclusiones

El modelo lakehouse permitió responder las cinco preguntas de negocio planteadas sin
necesidad de mover los datos a un motor distinto: las mismas tablas Delta que usamos para
la ingesta (capa bronce) y la limpieza (capa plata) sirvieron directamente para las consultas
analíticas, incluyendo la que requería aplanar el struct `metadata` del clickstream. Esto
confirma la decisión tomada en la sección 3: no hizo falta un motor NoSQL aparte para lo
semiestructurado, ni una base relacional aparte para lo tabular.

En cuanto a los resultados concretos: [destino/destinos que lideraron en ingresos, según la
Consulta 1] concentraron la mayor parte de los ingresos por reservas confirmadas, mientras
que la tasa de cancelación observada fue de [porcentaje aproximado, según la Consulta 3],
lo que consideramos [alto/bajo/razonable] para este tipo de plataforma. En el análisis de
clickstream, el dispositivo con más eventos registrados fue [dispositivo líder, según la
Consulta 4], lo que sugiere que el esfuerzo de UX debería priorizarse ahí.

Sobre la calidad de los datos: el filtro `is_deleted = false` en `reviews` y la deduplicación
por `booking_id` en `bookings` fueron suficientes para dejar la capa plata limpia; [si notaste
algo raro, descríbelo aquí — por ejemplo: nulos inesperados, fechas fuera de rango, montos en
cero, duplicados que no se detectaron con el filtro usado. Si no notaste nada raro, dejar:
"no se identificaron problemas de calidad adicionales a los ya previstos en el diseño"].

Si tuviéramos que empezar de nuevo, [ajustarían algo del diseño de las capas — por ejemplo:
particionar `silver_bookings` por mes de check-in para acelerar la Consulta 5, agregar más
validaciones de calidad en la capa plata, o versionar explícitamente el esquema del clickstream
antes de aplanarlo].

---

# 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
# 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Ronal Mosquera | *(completar, p. ej.: catálogo/esquema, capa bronce y consultas 1 y 3)* | *(SUSTENTA Y ABRE LA EXPLICACION)* |
| Federico López Torres | *(completar, p. ej.: capa plata, datos semiestructurados y propiedades del lakehouse)* | *(CIERRA Y EXPLICA-CODIGO)* |
| *(integrante 3, si aplica)* | | |

**Uso de asistentes de IA:** se usó un asistente de IA (Claude) para estructurar el notebook,
redactar el contexto del problema, la comparación de paradigmas y las consultas analíticas
sobre el dataset `samples.wanderbricks`,

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas